# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MEDOXIII/FlyRank-Internship-ML/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

### 1. My rule and its reason codes

**Rule in plain words.** I prioritize pages that are at least 180 days old and received at least 500 impressions in the trailing 90 days. Among these eligible pages, pages with more impressions rank first. This is a transparent refresh-review queue. It is not a prediction of future traffic and not proof that refreshing a page will improve performance.

**Reason codes the rule can output**

| Reason code | Meaning | How it is ranked |
|---|---|---|
| `stale_and_visible` | Age ≥ 180 days **and** impressions_90d ≥ 500 | Eligible; score = impressions_90d |
| `not_eligible_too_new` | Age < 180 days, but impressions ≥ 500 | Score 0 |
| `not_eligible_low_impressions` | Age ≥ 180 days, but impressions < 500 | Score 0 |
| `not_eligible_new_and_low_impressions` | Fails both thresholds | Score 0 |
| `missing_data` | Age or impressions is missing | Score 0; kept separate so missing data is not silently treated as "not eligible" |

**Design choices and limits**
- The two thresholds (180 days and 500 impressions) are judgment calls. They were not tuned on the decline label.
- The score uses only `content_age_days` and `impressions_90d`. The decline label, `trend_direction`, `trend_pct`, IDs, product/provider flags and 30-day windows are not scoring inputs. `content_id` is used only as a deterministic tie-breaker.
- Because the trend fields are excluded, this queue surfaces **visible, older pages**. It does not detect decline. Decline is used only afterwards, as a proxy label for evaluation.

In [1]:
from pathlib import Path
import re
import numpy as np
import pandas as pd

# Works when the notebook is run from the repository root or work/notebooks.
candidates = [
    Path("data/processed/refresh_feature_vector.csv"),
    Path("../../data/processed/refresh_feature_vector.csv"),
]
data_path = next((p for p in candidates if p.exists()), None)
if data_path is None:
    raise FileNotFoundError("Could not find data/processed/refresh_feature_vector.csv")

df = pd.read_csv(data_path)
raw_columns = list(df.columns)   # used later in the leakage check

required = {
    "content_id", "content_age_days", "impressions_90d",
    "is_declining_label", "trend_direction", "trend_pct",
}
missing_cols = required - set(df.columns)
assert not missing_cols, f"Missing required columns: {sorted(missing_cols)}"

# Data-contract checks
assert df["content_id"].is_unique, "content_id must be unique (one row per content item)"
for col in ["content_age_days", "impressions_90d"]:
    assert pd.api.types.is_numeric_dtype(df[col]), f"{col} must be numeric"
assert df["is_declining_label"].notna().all(), "Label has missing values; decide how to handle them first"

print(f"Rows: {len(df):,}")
print("\nMissing values in the two score inputs:")
print(df[["content_age_days", "impressions_90d"]].isna().sum().to_string())
print("\nDistribution of the score inputs (context for the thresholds):")
print(df[["content_age_days", "impressions_90d"]].describe().round(1).to_string())
print("\nLabel base rate:", round(df["is_declining_label"].astype(int).mean(), 3))
print("Label agrees with trend_direction == 'down':",
      round(df["trend_direction"].eq("down").eq(df["is_declining_label"].astype(bool)).mean(), 3))

Rows: 30,000

Missing values in the two score inputs:
content_age_days    0
impressions_90d     0

Distribution of the score inputs (context for the thresholds):
       content_age_days  impressions_90d
count           30000.0          30000.0
mean              256.2           5200.4
std               132.7          16838.0
min                90.0              1.0
25%               132.0             81.0
50%               236.0            731.0
75%               333.0           3615.2
max               564.0         517715.0

Label base rate: 0.542
Label agrees with trend_direction == 'down': 1.0


### 2. Build the ranked queue

The score is `impressions_90d` for eligible pages and 0 otherwise. All pages are ranked, so the queue covers the full dataset, and the reason codes separate eligible pages from the rest (including pages with missing inputs). Ties are broken deterministically by impressions, then age (older first), then `content_id`, which is used only as a tie-breaker and never as a score input.

The decline label is used only after scoring, to evaluate Precision@50 against the label base rate. The queue is written to `work/outputs/baseline_action_score.csv` and contains no label or trend columns.

**Result.** 9,929 of 30,000 pages are eligible (`stale_and_visible`). Precision@50 is 0.420, below the label base rate of 0.542 (see question 4 for baselines and interpretation).

In [2]:
AGE_THRESHOLD = 180
IMPRESSION_THRESHOLD = 500

age = df["content_age_days"]
imp = df["impressions_90d"]

missing_inputs = age.isna() | imp.isna()
old_enough = age.ge(AGE_THRESHOLD)        # NaN -> False
visible = imp.ge(IMPRESSION_THRESHOLD)    # NaN -> False

df["eligible"] = old_enough & visible
df["score"] = np.where(df["eligible"], imp, 0)
df["reason_code"] = np.select(
    [missing_inputs, ~old_enough & ~visible, ~old_enough, ~visible],
    ["missing_data",
     "not_eligible_new_and_low_impressions",
     "not_eligible_too_new",
     "not_eligible_low_impressions"],
    default="stale_and_visible",
)

print("Pages per reason code:")
print(df["reason_code"].value_counts().to_string())
assert df["eligible"].sum() > 0, "No eligible pages: revisit the thresholds"
print("\nImpressions among eligible pages:")
print(df.loc[df["eligible"], "impressions_90d"].describe().round(1).to_string())

# content_id is only a deterministic tie-breaker, never a score input.
ranked = df.sort_values(
    ["score", "impressions_90d", "content_age_days", "content_id"],
    ascending=[False, False, False, True],
).reset_index(drop=True)
ranked.insert(0, "rank", np.arange(1, len(ranked) + 1))

output_dir = Path("work/outputs")
output_dir.mkdir(parents=True, exist_ok=True)
queue_path = output_dir / "baseline_action_score.csv"
ranked[[
    "rank", "content_id", "score", "reason_code",
    "content_age_days", "impressions_90d",
]].to_csv(queue_path, index=False)

# Evaluation only (the label was not used for scoring)
k = min(50, len(ranked))
labels = df.set_index("content_id")["is_declining_label"].astype(int)
top_k_labels = ranked.head(k)["content_id"].map(labels)

if df["eligible"].sum() < k:
    print(f"\nWarning: only {int(df['eligible'].sum())} eligible pages; "
          f"the top {k} includes ineligible pages.")
print(f"\nQueue saved: {queue_path}")
print(f"Precision@{k}: {top_k_labels.mean():.3f}")
print(f"Base rate: {labels.mean():.3f}")

Pages per reason code:
reason_code
stale_and_visible                       9929
not_eligible_low_impressions            8057
not_eligible_too_new                    6797
not_eligible_new_and_low_impressions    5217

Impressions among eligible pages:
count      9929.0
mean       9409.8
std       22372.6
min         500.0
25%        1227.0
50%        2968.0
75%        8439.0
max      517715.0

Queue saved: work\outputs\baseline_action_score.csv
Precision@50: 0.420
Base rate: 0.542


## ### 3. Top-20 review

The top-20 list is a metadata-based review queue, not a final recommendation to edit every page. For each page the worksheet gives a suggested action, a reason code, a confidence note and what would make the pick wrong. Every top-20 page carries the same reason code (`stale_and_visible`) because the rule has a single eligible code, so the differences between rows come from age and impression level.

Confidence is preliminary because this notebook does not inspect page content, and the rule does not use update-history fields (the data includes `days_since_last_update`, which is not used in the score). In general a pick could be wrong if the page is already current, if its impressions are not relevant to the business, or if an update is not appropriate (for example an evergreen or archival page).

**Observations after reviewing the worksheet**
- All 20 pages are `stale_and_visible`, with ages of 211–537 days and 181k–518k impressions. None is older than 3 years (the oldest page in the dataset is 564 days), so the "very old / evergreen" caution never fires and the suggested action is the same for all 20.
- Every page is at or above the top-1% impressions level among eligible pages, so all 20 carry the hub/brand-page caution. This is the first thing to check: a page with hundreds of thousands of impressions may be a category, home or brand page rather than an article that needs a refresh.
- Ten pages are at least 360 days old and get a preliminary-medium confidence note. The other ten are 211–313 days old and get a preliminary-low note. Six of them (ranks 10, 12, 14, 16, 18, 20) are under 270 days old, close to the 180-day cutoff.
- Because the list is sorted purely by impressions, it is dominated by the highest-volume pages; age only decides who is eligible, not the order.

The worksheet is generated from metadata only. Its confidence notes and possible failure reasons are preliminary; the pages have not been individually reviewed for content or update history, so the `human_review_notes` column is intentionally left blank.

In [3]:
pd.set_option("display.max_colwidth", None)

top20 = ranked.head(20).copy()
if int(top20["eligible"].sum()) < 20:
    print("Warning: fewer than 20 eligible pages; some top-20 rows are not eligible.")

elig_imp = ranked.loc[ranked["eligible"], "impressions_90d"]
p99 = elig_imp.quantile(0.99)   # impressions level of the top 1% of eligible pages

def wrong_if(r):
    if not r["eligible"]:
        return "Not eligible under the rule; shown only because fewer than 20 pages qualify"
    notes = []
    if r["content_age_days"] >= 3 * 365:
        notes.append("very old: may be evergreen/archival, so age alone is weak evidence")
    if r["impressions_90d"] >= p99:
        notes.append("extreme impressions: may be a hub/brand page rather than a refresh target")
    if r["content_age_days"] < 1.5 * AGE_THRESHOLD:
        notes.append("only just past the age cutoff")
    notes.append("page may already be current or updated outside this data; "
                 "impressions may not matter to the business")
    return "; ".join(notes)

def confidence(r):
    if not r["eligible"]:
        return "None: not eligible under the rule"
    strong = (r["impressions_90d"] >= 2 * IMPRESSION_THRESHOLD
              and r["content_age_days"] >= 2 * AGE_THRESHOLD)
    return ("Preliminary-medium: high volume and clearly old; content and update history unchecked"
            if strong else
            "Preliminary-low: close to a threshold; verify before acting")

top20["suggested_action"] = np.select(
    [~top20["eligible"], top20["content_age_days"] >= 3 * 365],
    ["Do not review: not eligible under the rule",
     "Check whether the page is evergreen; if not, schedule a refresh review"],
    default="Review for possible refresh; do not edit automatically",
)
top20["confidence_note"] = top20.apply(confidence, axis=1)
top20["what_could_make_this_wrong"] = top20.apply(wrong_if, axis=1)
top20["human_review_notes"] = ""

review = top20[[
    "rank", "content_id", "reason_code", "content_age_days", "impressions_90d",
    "suggested_action", "confidence_note", "what_could_make_this_wrong",
    "human_review_notes",
]].copy()

review_path = output_dir / "baseline_top20_review.csv"
review.to_csv(review_path, index=False)
print(f"Top-20 review worksheet saved: {review_path}")
review

Top-20 review worksheet saved: work\outputs\baseline_top20_review.csv


,rank,content_id,reason_code,content_age_days,impressions_90d,suggested_action,confidence_note,what_could_make_this_wrong,human_review_notes
0,1,content_5fe46e04994d,stale_and_visible,537,517715,Review for possible refresh; do not edit automatically,Preliminary-medium: high volume and clearly old; content and update history unchecked,extreme impressions: may be a hub/brand page rather than a refresh target; page may already be current or updated outside this data; impressions may not matter to the business,
1,2,content_aaef01a50def,stale_and_visible,445,517109,Review for possible refresh; do not edit automatically,Preliminary-medium: high volume and clearly old; content and update history unchecked,extreme impressions: may be a hub/brand page rather than a refresh target; page may already be current or updated outside this data; impressions may not matter to the business,
2,3,content_8c19996aa890,stale_and_visible,445,509252,Review for possible refresh; do not edit automatically,Preliminary-medium: high volume and clearly old; content and update history unchecked,extreme impressions: may be a hub/brand page rather than a refresh target; page may already be current or updated outside this data; impressions may not matter to the business,
3,4,content_4c36c775b818,stale_and_visible,445,463103,Review for possible refresh; do not edit automatically,Preliminary-medium: high volume and clearly old; content and update history unchecked,extreme impressions: may be a hub/brand page rather than a refresh target; page may already be current or updated outside this data; impressions may not matter to the business,
4,5,content_2dba2b1f9536,stale_and_visible,299,443434,Review for possible refresh; do not edit automatically,Preliminary-low: close to a threshold; verify before acting,extreme impressions: may be a hub/brand page rather than a refresh target; page may already be current or updated outside this data; impressions may not matter to the business,
5,6,content_1a9e894be2e2,stale_and_visible,482,416180,Review for possible refresh; do not edit automatically,Preliminary-medium: high volume and clearly old; content and update history unchecked,extreme impressions: may be a hub/brand page rather than a refresh target; page may already be current or updated outside this data; impressions may not matter to the business,
6,7,content_2c2606c5d176,stale_and_visible,362,347399,Review for possible refresh; do not edit automatically,Preliminary-medium: high volume and clearly old; content and update history unchecked,extreme impressions: may be a hub/brand page rather than a refresh target; page may already be current or updated outside this data; impressions may not matter to the business,
7,8,content_db5989a78dd3,stale_and_visible,445,345111,Review for possible refresh; do not edit automatically,Preliminary-medium: high volume and clearly old; content and update history unchecked,extreme impressions: may be a hub/brand page rather than a refresh target; page may already be current or updated outside this data; impressions may not matter to the business,
8,9,content_9532f197bbc8,stale_and_visible,445,309192,Review for possible refresh; do not edit automatically,Preliminary-medium: high volume and clearly old; content and update history unchecked,extreme impressions: may be a hub/brand page rather than a refresh target; page may already be current or updated outside this data; impressions may not matter to the business,
9,10,content_8e7ba84a972b,stale_and_visible,224,288426,Review for possible refresh; do not edit automatically,Preliminary-low: close to a threshold; verify before acting,extreme impressions: may be a hub/brand page rather than a refresh target; only just past the age cutoff; page may already be current or updated outside this data; impressions may not matter to the business,


### 4. Weak picks + leakage check

**Weak picks.** Ten of the top 20 pages were not labeled declining in the observed window (7 labeled `stable` and 3 `up`), giving Precision@20 of 0.50. These are weak picks only relative to the notebook's proxy label; that does not necessarily mean they are poor editorial review candidates. According to the metadata flags, all ten have impressions at or above the eligible-page 99th percentile, so they may include high-volume hub or brand pages. Five of the six top-20 pages younger than 270 days were not labeled declining, compared with five of the fourteen older pages. These counts are small, and they are descriptive observations from this dataset, not evidence that age caused the difference. The worksheet is metadata-based, so the pages still need actual content review.

**Baselines.** In this run, Rule Precision@50 was 0.420, compared with a label base rate of 0.542 (lift 0.78). Impressions-only Precision@50 was also 0.420, which is expected because the rule ranks eligible pages by impressions, so the age gate is not what drives the result. Age-only Precision@50 was 0.760 (lift 1.40). Thus, on this proxy label and this dataset, the rule did not outperform the base rate, and age-only ranking had higher measured Precision@50. With only 50 pages, Precision@50 has a standard error of about 0.07 around the base rate, so the rule's shortfall is weak evidence on its own. These are observed comparisons, not evidence of future performance or proof that age is generally a better ranking signal. The rule was kept as specified rather than tuned to the label.

**Leakage check and limitations.** The score is reproduced exactly from only `content_age_days` and `impressions_90d`. The label, `trend_direction` and `trend_pct` are not scoring inputs, and neither are provider/model fields, `measurable_opportunity`, or any 30-day window (`impressions`, `clicks` and `sessions` for the last and previous 30 days). IDs are used only for joining or deterministic tie-breaking, and the saved queue contains no label or trend columns. However, the trailing-90-day metrics overlap the proxy-label window. Therefore, Precision@50 is a directional within-window comparison, not a future-prediction evaluation. The top-20 worksheet is an initial review aid, not a substitute for human review of page content; the notebook also did not use update-history fields such as `days_since_last_update`.


In [4]:
# ---- 1) Leakage check: the score must be reproducible from the allowed inputs only ----
SCORE_INPUTS = ["content_age_days", "impressions_90d"]
FORBIDDEN_EXACT = {
    "is_declining_label", "trend_direction", "trend_pct",
    "content_id", "client_id", "provider_used", "model_used",
    "measurable_opportunity",
    "impressions_last_30d", "clicks_last_30d", "sessions_last_30d",
    "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d",
}
FORBIDDEN_PATTERN = r"trend|label|30d|provider|model_used|client|product|flag|opportunity|future|next"

assert set(SCORE_INPUTS).isdisjoint(FORBIDDEN_EXACT), "Forbidden column in score inputs"
assert not [c for c in SCORE_INPUTS if re.search(FORBIDDEN_PATTERN, c, re.I)], "Risky column name in score inputs"

def score_from_inputs(frame):
    # frame contains ONLY the allowed columns, so touching any other column raises KeyError
    ok = (frame["content_age_days"].ge(AGE_THRESHOLD)
          & frame["impressions_90d"].ge(IMPRESSION_THRESHOLD))
    return np.where(ok, frame["impressions_90d"], 0)

assert np.array_equal(score_from_inputs(df[SCORE_INPUTS]), df["score"].to_numpy()), \
    "Saved score cannot be reproduced from the allowed inputs alone"

saved_cols = set(pd.read_csv(queue_path, nrows=0).columns)
assert saved_cols.isdisjoint(FORBIDDEN_EXACT - {"content_id"}), "Label/trend column leaked into the saved queue"

print("Columns in the data NOT used by the score:")
print([c for c in raw_columns if c not in SCORE_INPUTS])
print("\nLeakage check passed: the score is reproducible from content_age_days and impressions_90d only.")
print("Caution: trailing-90-day inputs overlap the proxy-label window.")

# ---- 2) Weak picks in the top 20 (relative to the proxy label) ----
label_of = lambda frame: frame["content_id"].map(labels).astype(int)
top20 = ranked.head(20)
p99 = ranked.loc[ranked["eligible"], "impressions_90d"].quantile(0.99)

def metadata_flags(r):
    flags = []
    if r["content_age_days"] >= 3 * 365:
        flags.append("very old (possibly evergreen)")
    if r["impressions_90d"] >= p99:
        flags.append("extreme impressions (possible hub/brand page)")
    if r["content_age_days"] < 1.5 * AGE_THRESHOLD:
        flags.append("just past the age cutoff")
    if r["impressions_90d"] < 2 * IMPRESSION_THRESHOLD:
        flags.append("near the impressions cutoff")
    return "; ".join(flags) if flags else "no metadata red flag; needs content review"

weak = top20.assign(proxy_label=label_of(top20)).query("proxy_label == 0").copy()
print(f"\nTop-20 picks not labeled declining by the proxy: {len(weak)} of 20")
if weak.empty:
    print("No top-20 picks disagree with the proxy label.")
else:
    weak["possible_reason"] = weak.apply(metadata_flags, axis=1)
    # trend_direction is shown for evaluation only; it was never used in the score
    print(weak[["rank", "content_id", "content_age_days", "impressions_90d",
                "trend_direction", "possible_reason"]].to_string(index=False))

# ---- 3) Precision@k vs base rate and simple baselines (directional, within-window) ----
by_impressions = df.sort_values(["impressions_90d", "content_id"], ascending=[False, True]).head(k)
by_age = df.sort_values(["content_age_days", "content_id"], ascending=[False, True]).head(k)

base_rate = labels.mean()
results = pd.DataFrame({
    "precision": {
        f"Rule Precision@{k}": label_of(ranked.head(k)).mean(),
        "Base rate (random expectation)": base_rate,
        f"Impressions-only top {k}": label_of(by_impressions).mean(),
        f"Age-only top {k}": label_of(by_age).mean(),
    }
})
results["lift_vs_base"] = results["precision"] / base_rate
print("\n", results.round(3).to_string())

Columns in the data NOT used by the score:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct', 'is_declining_label', 'log_impressions_90d', 'log_clicks_90d', 'log_sessions_90d', 'log_ai_sessions_90d', 'has_clicks', 'has_ai_sessions', 'measurable_opportunity']

Leakage check passed: the score is reproducible from conten

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.